In [ ]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
from rasterstats import zonal_stats
from rasterio.features import rasterize
from rasterio.plot import show
import matplotlib.pyplot as plt
import seaborn as sns

In [57]:
def analyse_fire_vegetation_impact(vegetation_tif_path, fire_shp_path):
    """
    Analyse which vegetation categories are most affected by fires.

    Parameters:
    -----------
    vegetation_tif_path : str
        Path to the vegetation raster TIF file
    fire_shp_path : str
        Path to the fire boundaries shapefile

    Returns:
    --------
    pd.DataFrame
        Summary statistics of vegetation categories affected by fires
    """

    # Load fire polygons
    fires = gpd.read_file(fire_shp_path)
    print(f"Loaded {len(fires)} fire polygons")

    # Check the CRS of both datasets
    if fires.crs is None:
        print("Fire shapefile has no CRS defined. Please set the CRS before proceeding.")
        return None

    with rasterio.open(vegetation_tif_path) as src:
        vegetation_crs = src.crs
        if vegetation_crs is None:
            print(
                "Vegetation raster has no CRS defined. Please set the CRS before proceeding.")
            return None
        print(f"Vegetation raster CRS: {vegetation_crs}")

    print(f"Fire vector CRS: {fires.crs}")

    # Reproject fires to hawaii CRS
    if fires.crs != vegetation_crs:
        print("Reprojecting fire polygons to match vegetation raster...")
        fires = fires.to_crs(vegetation_crs)

    # Calculate zonal statistics to get vegetation categories within fire areas
    stats = zonal_stats(
        fires.geometry,
        vegetation_tif_path,
        categorical=True,  # This will count pixels for each category
        geojson_out=True
    )

    # Process results
    vegetation_fire_data = []

    # https://rasterio.readthedocs.io/en/stable/topics/transforms.html
    # https://geog-510.gishub.org/book/geospatial/rasterio.html#affine-transform
    with rasterio.open(vegetation_tif_path) as src:
        pixel_area = abs(src.transform[0] * src.transform[4])  # Area per pixel

        for i, stat in enumerate(stats):
            if 'properties' in stat and stat['properties'] is not None:
                for veg_category, pixel_count in stat['properties'].items():
                    if isinstance(veg_category, (int, float)) and not np.isnan(float(veg_category)):
                        area = pixel_count * pixel_area
                        vegetation_fire_data.append({
                            'fire_id': i,
                            'vegetation_category': int(float(veg_category)),
                            'pixel_count': pixel_count,
                            'area_sq_meters': area,
                            'area_hectares': area / 10000  # Convert to hectares since qgis uses area_ha
                        })

    # Create DataFrame and summarize
    df = pd.DataFrame(vegetation_fire_data)

    if len(df) == 0:
        print("No vegetation data found within fire boundaries!")
        return None

    # group by vegetation category
    summary = df.groupby('vegetation_category').agg({
        'area_hectares': 'sum',
        'pixel_count': 'sum',
        'fire_id': 'nunique'  # Number of fires affecting this vegetation type
    }).reset_index()

    summary.columns = ['vegetation_category',
                       'total_area_hectares', 'total_pixels', 'num_fires_features']
    summary = summary.sort_values('total_area_hectares', ascending=False)

    # Calculate percentages
    summary['percentage_of_total_burned'] = (summary['total_area_hectares'] /
                                             summary['total_area_hectares'].sum() * 100)

    return summary


df = analyse_fire_vegetation_impact(
    r'../data\vegetation\LF2014_EVT_140_HI\Tif\hi_140evt.tif', r'../data\fires\fires_1999_2022.shp')  # type: ignore

Loaded 371 fire polygons
Vegetation raster CRS: ESRI:102007
Fire vector CRS: EPSG:4326
Reprojecting fire polygons to match vegetation raster...


In [58]:
# Get the unique vegetation categories for the vegetation raster
metadata = pd.read_csv(r'../data\vegetation\LF2014_EVT_140_HI\CSV_Data\hi_140evt.csv')
# Link the ClASSNAME to the Value Column
value_to_classname = dict(zip(metadata['VALUE'], metadata['CLASSNAME']))

# replace the numbers in df['vegetation_category'] with the class names
fire_veg_df = df.copy()
fire_veg_df['vegetation_category'] = fire_veg_df['vegetation_category'].map(value_to_classname)

fire_veg_df


,vegetation_category,total_area_hectares,total_pixels,num_fires_features,percentage_of_total_burned
30,Hawai'i Introduced Perennial Grassland,75863.52,842928,340,53.280103
29,Hawai'i Introduced Deciduous Shrubland,15447.33,171637,289,10.848895
4,Agriculture,8694.81,96609,82,6.106497
27,Hawai'i Introduced Dry Forest,7881.03,87567,249,5.534967
18,Hawai'i Montane-Subalpine Dry Shrubland,5991.75,66575,32,4.208097
25,Barren,4360.50,48450,109,3.062445
14,Hawai'i Lowland Dry Shrubland,3405.42,37838,95,2.391678
10,Hawai'i Lowland Dry Forest,2659.59,29551,104,1.867870
0,Developed-Open Space,2647.98,29422,264,1.859717
13,Hawai'i Montane-Subalpine Mesic Forest,2645.37,29393,23,1.857884
